In [109]:
import pandas as pd
import numpy as np

# 4)

In [110]:
df = pd.read_excel("inputs/Domestic_shares.xlsx", header=2)
df = df.drop(columns=['Unnamed: 0'])
df = df.set_index('Country')

In [111]:
df = df[df.index.isin(["USA", "CAN", "MEX", "CHN"])][[2000, 2014]]

In [112]:
countries = ['USA', 'CAN', 'MEX', 'CHN']
years = [2000, 2014]
epsilons = [4, 5]

rows = []

for c in countries:
    row = {'Country': c}

    for y in years:
        lam = df.loc[c, y]
        row[f'lambda_{y}'] = lam

        for e in epsilons:
            gft = (lam**(-1/e) - 1) * 100
            row[f'GFT_{y}_e{e}'] = gft

    rows.append(row)

out = pd.DataFrame(rows)

out = out.rename(columns={
    'lambda_2000': 'λii (2000)',
    'GFT_2000_e4': 'GFT (2000) ε=4',
    'GFT_2000_e5': 'GFT (2000) ε=5',
    'lambda_2014': 'λii (2014)',
    'GFT_2014_e4': 'GFT (2014) ε=4',
    'GFT_2014_e5': 'GFT (2014) ε=5',
})

out_display = out.copy()

for col in out_display.columns[1:]:
    out_display[col] = out_display[col].round(3)

out_display

,Country,λii (2000),GFT (2000) ε=4,GFT (2000) ε=5,λii (2014),GFT (2014) ε=4,GFT (2014) ε=5
0,USA,0.932,1.775,1.418,0.923,2.011,1.606
1,CAN,0.816,5.228,4.161,0.832,4.719,3.758
2,MEX,0.845,4.310,3.433,0.830,4.775,3.802
3,CHN,0.931,1.795,1.433,0.941,1.536,1.227


# 6)

In [113]:
df = pd.read_stata("inputs/WIOT2014_October16_ROW.dta")

df = df[df['Country']!='TOT']

df = df.groupby("Country").sum().reset_index()

for country in list(df['Country']):
    df[country] = df[[col for col in df.columns if col.startswith("v" + country)]].sum(axis=1)

df = df[["Country"] + list(df['Country'])]

In [114]:
df2 = df.iloc[:,~(df.columns.isin(['USA','CHN','CAN','MEX']))]
df3 = df2['Country'].copy().to_frame()
df['ROW'] = np.sum(df2.iloc[:,1:],axis = 1)

In [115]:
exclude = ["USA", "CHN", "CAN", "MEX"]

subset = df.loc[~df["Country"].isin(exclude)]

new_row = subset.sum(axis=0)

agg_df = new_row.to_frame().T

agg_df = agg_df[df.columns]

country_col = df.columns[0]
agg_df[country_col] = "AGG_OTHERS"

In [116]:
us_2014 = {"USA": [
    df[df["Country"] == "USA"]["USA"].values[0] / df["USA"].sum(),
    df[df["Country"] == "CHN"]["USA"].values[0] / df["USA"].sum(),
    df[df["Country"] == "MEX"]["USA"].values[0] / df["USA"].sum(),
    df[df["Country"] == "CAN"]["USA"].values[0] / df["USA"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["USA"].values[0] / df["USA"].sum()
],
     "CHN": [
    df[df["Country"] == "USA"]["CHN"].values[0] / df["CHN"].sum(),
    df[df["Country"] == "CHN"]["CHN"].values[0] / df["CHN"].sum(),
    df[df["Country"] == "MEX"]["CHN"].values[0] / df["CHN"].sum(),
    df[df["Country"] == "CAN"]["CHN"].values[0] / df["CHN"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["CHN"].values[0] / df["CHN"].sum()
],
     "MEX": [
    df[df["Country"] == "USA"]["MEX"].values[0] / df["MEX"].sum(),
    df[df["Country"] == "CHN"]["MEX"].values[0] / df["MEX"].sum(),
    df[df["Country"] == "MEX"]["MEX"].values[0] / df["MEX"].sum(),
    df[df["Country"] == "CAN"]["MEX"].values[0] / df["MEX"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["MEX"].values[0] / df["MEX"].sum()
],
     "CAN": [
    df[df["Country"] == "USA"]["CAN"].values[0] / df["CAN"].sum(),
    df[df["Country"] == "CHN"]["CAN"].values[0] / df["CAN"].sum(),
    df[df["Country"] == "MEX"]["CAN"].values[0] / df["CAN"].sum(),
    df[df["Country"] == "CAN"]["CAN"].values[0] / df["CAN"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["CAN"].values[0] / df["CAN"].sum()
]}

us_2000 = np.array([[0.932,0.003,0.007,0.012,0.046],
                [0.004,0.931,0.0000001,0.001,0.064],
                [0.097,0.002,0.845,0.003,0.054],
                [0.114,0.004,0.005,0.816,0.061]])

df_l2000 = pd.DataFrame(us_2000, index=["USA", "CHN", "MEX", "CAN"], columns=["USA", "CHN", "MEX", "CAN", "ROW"])

df_l2014 = pd.DataFrame(us_2014, index=["USA", "CHN", "MEX", "CAN", "ROW"]).T

In [117]:
## Expenditure shares 2000

df_l2000

,USA,CHN,MEX,CAN,ROW
USA,0.932,0.003,7.000000e-03,0.012,0.046
CHN,0.004,0.931,1.000000e-07,0.001,0.064
MEX,0.097,0.002,8.450000e-01,0.003,0.054
CAN,0.114,0.004,5.000000e-03,0.816,0.061


In [121]:
## Expenditure shares 2014

df_l2014.round(3)

,USA,CHN,MEX,CAN,ROW
USA,0.923,0.011,0.009,0.011,0.046
CHN,0.004,0.941,0.000,0.001,0.055
MEX,0.084,0.018,0.830,0.004,0.064
CAN,0.090,0.015,0.006,0.832,0.057


# 7)

![image](formula.png)

In [122]:
## Individual contributions

e = 5.0

l2014 = df_l2014.to_numpy()

l2000 = df_l2000.to_numpy()

t7 = np.full(shape=(4,5),fill_value=np.nan)

for i in range(0,4):
    for j in range(0,5):
        arr = np.concatenate([np.arange(0, i), np.arange(i + 1, 5)])
        t7[i,j] = 1/e*l2000[i,j]/l2000[i,i]*(l2014[i,j]-l2000[i,j])/l2000[i,j]

df7 = pd.DataFrame(t7,columns=['US Cont','CHN Cont','MEX Cont','CAN Cont','ROW Cont'],index=['USA','CHN','MEX','CAN'])

df7 = df7[["US Cont", "CAN Cont", "MEX Cont", "CHN Cont", "ROW Cont"]]

df7 = df7.reindex(["USA", "CAN", "MEX", "CHN"])

df7.round(4)

,US Cont,CAN Cont,MEX Cont,CHN Cont,ROW Cont
USA,-0.0018,-0.0002,0.0003,0.0017,-0.0000
CAN,-0.0058,0.0038,0.0003,0.0028,-0.0011
MEX,-0.0031,0.0002,-0.0036,0.0038,0.0024
CHN,-0.0001,-0.0001,0.0001,0.0021,-0.0020


In [123]:
## GFT and contributions as ratio

e = 5.0

l2014 = df_l2014.to_numpy()

l2000 = df_l2000.to_numpy()

t7 = np.full(shape=(4,6),fill_value=np.nan)

for i in range(0,4):
    arr = np.concatenate([np.arange(0, i), np.arange(i + 1, 5)])
    t7[i,0] = 1/e*np.sum(l2000[i,arr]/l2000[i,i]*(l2014[i,arr]-l2000[i,arr])/l2000[i,arr])

for i in range(0,4):
    for j in range(0,5):
            if j == i:
                 continue
            t7[i,j+1] = 1/e*l2000[i,j]/l2000[i,i]*(l2014[i,j]-l2000[i,j])/l2000[i,j]/t7[i,0]

df7 = pd.DataFrame(t7,columns=['GFT','US Cont','CHN Cont','MEX Cont','CAN Cont','ROW Cont'],index=['USA','CHN','MEX','CAN'])

df7 = df7[["GFT", "US Cont", "CAN Cont", "MEX Cont", "CHN Cont", "ROW Cont"]]

df7 = df7.reindex(["USA", "CAN", "MEX", "CHN"])

df7.round(4)

,GFT,US Cont,CAN Cont,MEX Cont,CHN Cont,ROW Cont
USA,0.0018,NaN,-0.1002,0.1755,0.9405,-0.0159
CAN,-0.0038,1.5243,NaN,-0.0714,-0.7297,0.2769
MEX,0.0034,-0.9095,0.0619,NaN,1.1306,0.7170
CHN,-0.0021,0.0410,0.0434,-0.0240,NaN,0.9396
